In [1]:
import sys
sys.path.insert(0, "/home/jovyan/project/work")

from lakehouse_spark import new_session
spark = new_session("session6_submission_checks")
spark.sparkContext.setLogLevel("WARN")

print("Spark ready")

Spark ready


In [2]:
import subprocess
subprocess.run([sys.executable, "/home/jovyan/project/work/bronze.py"], check=True)
subprocess.run([sys.executable, "/home/jovyan/project/work/silver.py"], check=True)
print("CDC bronze+silver done")

CDC bronze+silver done


In [15]:
import subprocess

def run_script(path):
    r = subprocess.run([sys.executable, path], text=True, capture_output=True)
    print(f"\n== {path} ==")
    print("exit:", r.returncode)
    if r.stdout:
        print("--- stdout (tail) ---")
        print("\n".join(r.stdout.splitlines()[-20:]))
    if r.stderr:
        print("--- stderr (tail) ---")
        print("\n".join(r.stderr.splitlines()[-40:]))
    return r.returncode

rc1 = run_script("/home/jovyan/project/work/taxi_bronze.py")
rc2 = run_script("/home/jovyan/project/work/taxi_silver.py")

if rc1 == 0 and rc2 == 0:
    print("Taxi bronze+silver done")
else:
    raise RuntimeError("Taxi build failed; inspect stderr above.")


== /home/jovyan/project/work/taxi_bronze.py ==
exit: 0
--- stdout (tail) ---
:: loading settings :: url = jar:file:/usr/local/spark-3.5.0-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml
Bronze taxi trips: 7052769
--- stderr (tail) ---
	org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.0 from central in [default]
	org.lz4#lz4-java;1.8.0 from central in [default]
	org.slf4j#slf4j-api;2.0.7 from central in [default]
	org.xerial.snappy#snappy-java;1.1.10.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   13  |   0   |   0   |   0   ||   13  |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent

In [16]:
spark.sql("DESCRIBE lakehouse.taxi.silver_trips").show(truncate=False)
spark.sql("SELECT PULocationID, pickup_zone, dropoff_zone, trip_duration_minutes, trip_speed_kmh FROM lakehouse.taxi.silver_trips LIMIT 10").show(truncate=False)

+---------------------+---------+-------+
|col_name             |data_type|comment|
+---------------------+---------+-------+
|trip_id              |bigint   |NULL   |
|PULocationID         |int      |NULL   |
|DOLocationID         |int      |NULL   |
|fare_amount          |double   |NULL   |
|tip_amount           |double   |NULL   |
|payment_type         |int      |NULL   |
|trip_distance        |double   |NULL   |
|tpep_pickup_datetime |timestamp|NULL   |
|tpep_dropoff_datetime|timestamp|NULL   |
|trip_duration_minutes|double   |NULL   |
|trip_speed_kmh       |double   |NULL   |
|pickup_zone          |string   |NULL   |
|dropoff_zone         |string   |NULL   |
+---------------------+---------+-------+

+------------+-----------------------------+------------------------+---------------------+------------------+
|PULocationID|pickup_zone                  |dropoff_zone            |trip_duration_minutes|trip_speed_kmh    |
+------------+-----------------------------+-------------------

In [8]:
spark.sql("SHOW NAMESPACES IN lakehouse").show(truncate=False)
    
for ns in ["cdc", "taxi"]:
    try:
        print(f"\nTables in lakehouse.{ns}:")
        spark.sql(f"SHOW TABLES IN lakehouse.{ns}").show(truncate=False)
    except Exception as e:
        print(f"lakehouse.{ns} not ready yet:", e)


+---------+
|namespace|
+---------+
|cdc      |
|taxi     |
+---------+


Tables in lakehouse.cdc:
+---------+----------------+-----------+
|namespace|tableName       |isTemporary|
+---------+----------------+-----------+
|cdc      |bronze_customers|false      |
|cdc      |silver_customers|false      |
+---------+----------------+-----------+


Tables in lakehouse.taxi:
+---------+------------+-----------+
|namespace|tableName   |isTemporary|
+---------+------------+-----------+
|taxi     |bronze_trips|false      |
|taxi     |silver_trips|false      |
+---------+------------+-----------+



In [9]:
spark.sql("SELECT count(*) AS cdc_cnt FROM lakehouse.cdc.silver_customers").show()
spark.sql("SELECT count(*) AS taxi_cnt FROM lakehouse.taxi.silver_trips").show()

+-------+
|cdc_cnt|
+-------+
|     12|
+-------+

+--------+
|taxi_cnt|
+--------+
| 6564677|
+--------+



In [10]:
# Task 3: gold_tipping_behavior prototype output (zone x hour)
spark.sql("""
WITH tip_base AS (
  SELECT
    t.PULocationID,
    hour(t.tpep_pickup_datetime) AS pickup_hour,
    t.fare_amount,
    t.tip_amount,
    t.payment_type,
    CASE WHEN t.fare_amount > 0 THEN (t.tip_amount / t.fare_amount) * 100 END AS tip_pct
  FROM lakehouse.taxi.silver_trips t
  WHERE t.fare_amount > 0
),
agg AS (
  SELECT
    PULocationID,
    pickup_hour,
    AVG(tip_amount) AS avg_tip_amount,
    AVG(tip_pct) AS avg_tip_pct,
    100.0 * AVG(CASE WHEN coalesce(tip_amount,0) = 0 THEN 1 ELSE 0 END) AS pct_zero_tip,
    100.0 * AVG(CASE WHEN tip_pct > 20 THEN 1 ELSE 0 END) AS pct_high_tip
  FROM tip_base
  GROUP BY PULocationID, pickup_hour
)
SELECT *
FROM agg
ORDER BY avg_tip_pct DESC
LIMIT 15
""").show(truncate=False)

+------------+-----------+------------------+------------------+------------------+------------------+
|PULocationID|pickup_hour|avg_tip_amount    |avg_tip_pct       |pct_zero_tip      |pct_high_tip      |
+------------+-----------+------------------+------------------+------------------+------------------+
|265         |15         |5.380641025641026 |5135.390011008078 |56.41025641025641 |24.358974358974358|
|1           |20         |49.666666666666664|1408.1560283687943|33.33333333333333 |66.66666666666666 |
|8           |22         |32.5              |127.95275590551182|50.0              |50.0              |
|246         |14         |3.0254578328813135|123.47657351286257|23.29846312798657 |62.50807180679323 |
|25          |6          |5.270241935483871 |103.64402312566246|73.38709677419355 |23.387096774193548|
|194         |4          |17.5              |91.62303664921465 |50.0              |50.0              |
|128         |13         |15.115            |89.14870689655173 |0.0      

In [13]:
# Task 3 question: which zone+hour gets the best tips?
spark.sql("""
WITH tip_base AS (
  SELECT
    t.PULocationID,
    hour(t.tpep_pickup_datetime) AS pickup_hour,
    CASE WHEN t.fare_amount > 0 THEN (t.tip_amount / t.fare_amount) * 100 END AS tip_pct
  FROM lakehouse.taxi.silver_trips t
  WHERE t.fare_amount > 0
)
SELECT
  PULocationID,
  pickup_hour,
  AVG(tip_pct) AS avg_tip_pct
FROM tip_base
GROUP BY PULocationID, pickup_hour
ORDER BY avg_tip_pct DESC
LIMIT 1
""").show(truncate=False)

+------------+-----------+-----------------+
|PULocationID|pickup_hour|avg_tip_pct      |
+------------+-----------+-----------------+
|265         |15         |5135.390011008078|
+------------+-----------+-----------------+



In [14]:
# Optional: driver rating vs tip correlation (only if silver_drivers exists)
try:
    spark.sql("""
    WITH driver_tip AS (
      SELECT
        t.driver_id,
        AVG(CASE WHEN t.fare_amount > 0 THEN (t.tip_amount / t.fare_amount) * 100 END) AS avg_tip_pct
      FROM lakehouse.taxi.silver_trips t
      GROUP BY t.driver_id
    )
    SELECT d.driver_id, d.rating, dt.avg_tip_pct
    FROM driver_tip dt
    JOIN lakehouse.cdc.silver_drivers d ON dt.driver_id = d.driver_id
    ORDER BY d.rating DESC
    LIMIT 20
    """).show(truncate=False)
except Exception as e:
    print("Driver correlation skipped:", e)
    print("(This is expected if driver_id/silver_drivers is not built yet.)")

Driver correlation skipped: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `t`.`driver_id` cannot be resolved. Did you mean one of the following? [`t`.`trip_id`, `t`.`tip_amount`, `t`.`fare_amount`, `t`.`payment_type`, `t`.`PULocationID`].; line 4 pos 8;
'WithCTE
:- 'CTERelationDef 7, false
:  +- 'SubqueryAlias driver_tip
:     +- 'Aggregate ['t.driver_id], ['t.driver_id, avg(CASE WHEN (fare_amount#432 > cast(0 as double)) THEN ((tip_amount#433 / fare_amount#432) * cast(100 as double)) END) AS avg_tip_pct#429]
:        +- SubqueryAlias t
:           +- SubqueryAlias lakehouse.taxi.silver_trips
:              +- RelationV2[trip_id#430L, PULocationID#431, fare_amount#432, tip_amount#433, payment_type#434, trip_distance#435, tpep_pickup_datetime#436] lakehouse.taxi.silver_trips lakehouse.taxi.silver_trips
+- 'GlobalLimit 20
   +- 'LocalLimit 20
      +- 'Sort ['d.rating DESC NULLS LAST], true
         +- 'Project ['d.driver_id, 'd.rating, 'dt.avg_tip_pct]
   